ICD-10 coding pipeline setup — source data, schemas, and reference tables
*Co-authored with CoCo*

# 00: Pipeline Setup

Prepares all source data, schemas, and reference tables for the ICD-10 coding pipeline.

**Run this notebook once** before running Steps 1–3.

**Source data:** `CHART_REVIEW_DB.RAW.ENCOUNTERS` (1,067 Aetna JSON chart reviews)
**Ground truth:** `CHART_REVIEW_DB.RAW.AETNA_GROUND_TRUTH` (2,112 validated ICD-10 assignments)
**Reference:** `ICD10_CODING_APP.ICD10_REF` (74k codes, HCC mappings, dual-code pairs)

In [ ]:
%%sql -r ctx
-- Context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICD10_CODING_APP;
USE WAREHOUSE COMPUTE_WH;
ALTER SESSION SET QUERY_TAG = 'icd10_v2:setup';

In [ ]:
%%sql -r schemas
-- Ensure schemas exist
CREATE SCHEMA IF NOT EXISTS ICD10_CODING_APP.PROCESSING;
CREATE SCHEMA IF NOT EXISTS ICD10_CODING_APP.MATCHING;
CREATE SCHEMA IF NOT EXISTS ICD10_CODING_APP.EXPERIMENTS;
CREATE SCHEMA IF NOT EXISTS ICD10_CODING_APP.ICD10_REF;

---
## Load Source Data

Extract structured sections from Aetna JSON encounters. Each encounter's RAW_RECORD contains nested fields for diagnoses, narratives, problems, etc.

In [ ]:
%%sql -r load_sections
-- Load encounter JSON data with all diagnosis-relevant fields extracted
CREATE OR REPLACE TABLE PROCESSING.DOCUMENT_SECTIONS AS
SELECT
    e.SOURCE_FILE AS FILE_NAME,
    e.RAW_RECORD::VARCHAR AS PARSED_TEXT,
    CONCAT(
        'Diagnosis Codes: ', COALESCE(e.DIAGNOSIS_CODES::VARCHAR, '[]'), CHR(10),
        'Visit Diagnoses: ', COALESCE(e.RAW_RECORD:Encounters_DiagnosisVisit::VARCHAR, '[]'), CHR(10),
        'Active Problems: ', COALESCE(e.RAW_RECORD:ActiveProblems::VARCHAR, '[]'), CHR(10),
        'Discharge Diagnoses: ', COALESCE(e.RAW_RECORD:DischargeDiagnoses::VARCHAR, '[]'), CHR(10),
        'Admitting Diagnoses: ', COALESCE(e.RAW_RECORD:AdmittingDiagnoses::VARCHAR, '[]'), CHR(10),
        'Resolved Problems: ', COALESCE(e.RAW_RECORD:ResolvedProblems::VARCHAR, '[]'), CHR(10),
        'Narratives: ', COALESCE(e.NARRATIVES::VARCHAR, '[]')
    ) AS DIAGNOSIS_SECTION,
    CONCAT(
        COALESCE(e.RAW_RECORD:VisitDiagnosis_Narrative::VARCHAR, ''), CHR(10),
        COALESCE(e.RAW_RECORD:ActiveProblems_Narrative::VARCHAR, ''), CHR(10),
        COALESCE(e.RAW_RECORD:DischargeDiagnosis_Narrative::VARCHAR, '')
    ) AS HPI_SECTION,
    COALESCE(e.RAW_RECORD:ResolvedProblems_Narrative::VARCHAR, NULL) AS PMH_SECTION,
    CASE
        WHEN e.RAW_RECORD:Encounters_EncounterCodesActCode::VARCHAR IN ('IMP', 'ACUTE') THEN 'INPATIENT'
        WHEN e.RAW_RECORD:Encounters_EncounterCodesActCode::VARCHAR = 'EMER' THEN 'ED'
        ELSE 'OUTPATIENT'
    END AS ENCOUNTER_SETTING
FROM CHART_REVIEW_DB.RAW.ENCOUNTERS e
WHERE e.RAW_RECORD IS NOT NULL;

In [ ]:
%%sql -r verify_source
-- Verify source data
SELECT
    COUNT(*) AS TOTAL_ENCOUNTERS,
    COUNT(DISTINCT UPPER(REGEXP_SUBSTR(FILE_NAME, 'eppmra[0-9]+', 1, 1, 'i'))) AS DISTINCT_CHASES,
    ROUND(AVG(LEN(DIAGNOSIS_SECTION)), 0) AS AVG_DX_SECTION_LEN,
    COUNT_IF(ENCOUNTER_SETTING = 'OUTPATIENT') AS OUTPATIENT,
    COUNT_IF(ENCOUNTER_SETTING = 'INPATIENT') AS INPATIENT
FROM PROCESSING.DOCUMENT_SECTIONS;

---
## Reference Tables

Dual-code pairs, enriched ICD-10 corpus, and search service.

In [ ]:
%%sql -r dual_codes
-- Dual-code pairs (etiology/manifestation relationships)
CREATE TABLE IF NOT EXISTS ICD10_REF.DUAL_CODE_PAIRS (
    ETIOLOGY_CODE VARCHAR(10),
    MANIFESTATION_CODE VARCHAR(10),
    RELATIONSHIP VARCHAR(100),
    DESCRIPTION VARCHAR(500)
);

MERGE INTO ICD10_REF.DUAL_CODE_PAIRS t
USING (
    SELECT column1 AS ETIOLOGY_CODE, column2 AS MANIFESTATION_CODE,
           column3 AS RELATIONSHIP, column4 AS DESCRIPTION
    FROM VALUES
    ('E11.21', 'N08.3', 'diabetic_nephropathy', 'Type 2 DM with diabetic nephropathy'),
    ('E11.22', 'N08.3', 'diabetic_ckd', 'Type 2 DM with diabetic CKD'),
    ('E11.31', 'H36.0', 'diabetic_retinopathy', 'Type 2 DM with background retinopathy'),
    ('E11.36', 'H28.0', 'diabetic_cataract', 'Type 2 DM with diabetic cataract'),
    ('E11.40', 'G99.0', 'diabetic_neuropathy', 'Type 2 DM with neuropathy unspecified'),
    ('E11.42', 'G63.2', 'diabetic_polyneuropathy', 'Type 2 DM with polyneuropathy'),
    ('E11.43', 'G59.0', 'diabetic_mononeuropathy', 'Type 2 DM with mononeuropathy'),
    ('E11.51', 'I79.2', 'diabetic_pvd', 'Type 2 DM with peripheral angiopathy'),
    ('E10.21', 'N08.3', 'diabetic_nephropathy_t1', 'Type 1 DM with nephropathy'),
    ('E10.42', 'G63.2', 'diabetic_polyneuropathy_t1', 'Type 1 DM with polyneuropathy'),
    ('A17.0', 'G01', 'tuberculous_meningitis', 'Tuberculous meningitis'),
    ('M32.1', 'N08.5', 'lupus_nephritis', 'SLE with glomerular disease')
) s ON t.ETIOLOGY_CODE = s.ETIOLOGY_CODE AND t.MANIFESTATION_CODE = s.MANIFESTATION_CODE
WHEN NOT MATCHED THEN INSERT VALUES (s.ETIOLOGY_CODE, s.MANIFESTATION_CODE, s.RELATIONSHIP, s.DESCRIPTION);

In [ ]:
%%sql -r enriched
-- Enriched ICD-10 corpus (for Cortex Search)
CREATE OR REPLACE TABLE ICD10_REF.ICD10_CODES_ENRICHED AS
SELECT
    c.ICD10_CODE,
    c.SHORT_DESCRIPTION,
    c.LONG_DESCRIPTION,
    c.CATEGORY_CODE,
    c.CHAPTER,
    h.HCC_CATEGORY,
    h.HCC_DESCRIPTION,
    h.RAF_COEFFICIENT,
    CASE WHEN dp.ETIOLOGY_CODE IS NOT NULL THEN TRUE ELSE FALSE END AS IS_DUAL_CODE_ETIOLOGY,
    CASE WHEN dp2.MANIFESTATION_CODE IS NOT NULL THEN TRUE ELSE FALSE END AS IS_DUAL_CODE_MANIFESTATION,
    CONCAT(
        c.ICD10_CODE, ' ',
        COALESCE(c.LONG_DESCRIPTION, c.SHORT_DESCRIPTION), ' ',
        COALESCE(c.CHAPTER, ''), ' ',
        COALESCE(h.HCC_DESCRIPTION, '')
    ) AS SEARCH_TEXT
FROM ICD10_REF.ICD10_CODES c
LEFT JOIN (
    SELECT ICD10_CODE, HCC_CATEGORY, HCC_DESCRIPTION, RAF_COEFFICIENT
    FROM ICD10_REF.HCC_MAPPINGS
    QUALIFY ROW_NUMBER() OVER (PARTITION BY ICD10_CODE ORDER BY RAF_COEFFICIENT DESC) = 1
) h ON c.ICD10_CODE = h.ICD10_CODE
LEFT JOIN (SELECT DISTINCT ETIOLOGY_CODE FROM ICD10_REF.DUAL_CODE_PAIRS) dp ON c.ICD10_CODE = dp.ETIOLOGY_CODE
LEFT JOIN (SELECT DISTINCT MANIFESTATION_CODE FROM ICD10_REF.DUAL_CODE_PAIRS) dp2 ON c.ICD10_CODE = dp2.MANIFESTATION_CODE;

In [ ]:
%%sql -r verify_enriched
-- Verify enriched corpus
SELECT
    COUNT(*) AS TOTAL_CODES,
    COUNT_IF(HCC_CATEGORY IS NOT NULL) AS HCC_RELEVANT,
    COUNT_IF(IS_DUAL_CODE_ETIOLOGY) AS DUAL_ETIOLOGIES,
    COUNT_IF(IS_DUAL_CODE_MANIFESTATION) AS DUAL_MANIFESTATIONS
FROM ICD10_REF.ICD10_CODES_ENRICHED;

---
## Verify Ground Truth Access

In [ ]:
%%sql -r gt_summary
-- Ground truth summary
SELECT
    COUNT(*) AS TOTAL_GT_ROWS,
    COUNT(DISTINCT CHASE_ID) AS DISTINCT_CHASES,
    COUNT(DISTINCT DIAGNOSIS_CODE) AS DISTINCT_CODES,
    COUNT(DISTINCT CHASE_ID || DIAGNOSIS_CODE) AS DISTINCT_CHASE_CODE_PAIRS
FROM CHART_REVIEW_DB.RAW.AETNA_GROUND_TRUTH
WHERE ICD_CODE_DISPOSITION = 'ADD';

---
## Setup Complete

| Asset | Location | Description |
|-------|----------|-------------|
| Source encounters | `PROCESSING.DOCUMENT_SECTIONS` | 1,067 encounters with extracted sections |
| ICD-10 codes | `ICD10_REF.ICD10_CODES_ENRICHED` | 74k codes with HCC + search text |
| Dual-code pairs | `ICD10_REF.DUAL_CODE_PAIRS` | Etiology/manifestation relationships |
| Ground truth | `CHART_REVIEW_DB.RAW.AETNA_GROUND_TRUTH` | 2,112 validated code assignments |

**Next:** Run `01_extraction.ipynb`